# Silver: Eventos

Preparação dos dados da bronze antes de salvar na silver. Sem cálculos de negócio: só tipagem, padronização, deduplicação e validação.

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
SOURCE_TABLE = "projetos.bronze.events"
TARGET_TABLE = "projetos.silver.eventos_colaboradores"

REQUIRED_COLUMNS = [
    "event_id", "worker_id"
]

In [0]:
df = spark.table(SOURCE_TABLE)

## Data Quality

In [0]:
rename_map = {
    "Event_ID": "event_id",
    "Worker_ID": "worker_id",
    "Business_Process": "business_process",
    "Event_Reason": "event_reason",
    "Effective_Date": "effective_date",
    "Entered_On": "entered_on",
    "Status": "status",
    "Initiated_By": "initiated_by",
    "Completed_On": "completed_on",
    "From_Job_Profile": "from_job_profile",
    "To_Job_Profile": "to_job_profile",
    "From_Supervisory_Organization": "from_supervisory_organization",
    "To_Supervisory_Organization": "to_supervisory_organization",
    "From_Location": "from_location",
    "To_Location": "to_location",
    "From_Base_Pay": "from_base_pay",
    "To_Base_Pay": "to_base_pay",
    "Currency": "currency",
    "Comment": "comment",
}

existing_columns = set(df.columns)

for old_name, new_name in rename_map.items():
    if old_name in existing_columns and old_name != new_name:
        df = df.withColumnRenamed(old_name, new_name)

Padronizar textos: remove espaços e transforma "null" / vazio em NULL de verdade

In [0]:
NULL_TOKENS = ["null", "NULL", "Null", ""]

string_cols = [f.name for f in df.schema.fields
               if f.dataType.typeName() == "string" and not f.name.startswith("_")]

for c in string_cols:
    limpo = F.trim(F.col(c))
    df = df.withColumn(c, F.when(limpo.isin(NULL_TOKENS) | limpo.isNull(), F.lit(None))
                           .otherwise(limpo))


Tipagem

In [0]:
df = df.withColumn('effective_date', F.to_date('effective_date', 'yyyy-MM-dd'))

Metadados de ingestão

In [0]:
df = df.withColumn('source_file', df._file_metadata["file_name"])
df = df.withColumn('file_modification_time', df._file_metadata["file_modification_time"])
df = df.drop("_file_metadata", '_gdrive_metadata')

Validação: falha cedo em vez de gravar lixo na silver

In [0]:
colunas_faltando = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if colunas_faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {colunas_faltando}")

chaves_nulas = df.where(F.col("worker_id").isNull()).count()
if chaves_nulas > 0:
    raise ValueError(f"{chaves_nulas} registros sem worker_id")

### Save

In [0]:
source = df 
target_table = DeltaTable.forName(spark, TARGET_TABLE)

target_table.alias("target")\
    .merge(source.alias("source"), condition="target.event_id = source.event_id AND target.file_modification_time > source.file_modification_time")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()